In [1]:
import pandas as pd
import numpy as np

In [2]:
PATH_VAL_CLIM = 'Valores_Climatologicos_1970_2024_con_Coordenadas.csv'

In [3]:
COLS_NOMBRES = [
    'indicativo',
    'nombre',
    'provincia',
]

COLS_INTS = [
    'altitud'
]

COLS_FLOATS = [
    'tmed',
    'prec',
    'tmin',
    'tmax',
    'velmedia',
    'sol',
    'presMax',
    'presMin',
    'hrMedia',
    'dir',
    'racha',
    'hrMax',
    'hrMin'
]

COLS_NUMS = COLS_INTS + COLS_FLOATS

COL_FECHA = 'fecha'
COLS_HORAS = [
    'horaPresMax',
    'horaPresMin',
    'horatmin',
    'horatmax',
    'horaracha',
    'horaHrMax',
    'horaHrMin'
]

COLS_COORDS = [
    'lat',
    'lon'
]

In [4]:
df = pd.read_csv(PATH_VAL_CLIM,
                 sep=';', decimal = ',',
                 dtype = {nombre: 'string' for nombre in COLS_NOMBRES} |
                         {entero: 'Int64' for entero in COLS_INTS} |
                         {coordenada: 'Float64' for coordenada in COLS_COORDS} |
                         {otros: 'object' for otros in COLS_FLOATS + COLS_HORAS},
                         # {decimal: 'Float64' for decimal in COLS_FLOATS if decimal != 'prec'},
                 parse_dates = [COL_FECHA])

In [5]:
display(df)

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,tmax,velmedia,...,horatmax,dir,racha,horaracha,hrMax,horaHrMax,hrMin,horaHrMin,lon,lat
0,1970-01-01,C249I,FUERTEVENTURA AEROPUERTO,LAS PALMAS,25,"19,8","0,0","16,0","23,5","6,7",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-13.863056,28.444722
1,1970-01-01,1679A,MONFORTE DE LEMOS,LUGO,291,"4,0","0,0","0,0","8,0",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-7.510833,42.531667
2,1970-01-01,2462,PUERTO DE NAVACERRADA,MADRID,1893,"-5,0","0,4","-7,0","-3,0","3,1",...,15:30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.010556,40.793056
3,1970-01-01,1212E,ASTURIAS AEROPUERTO,ASTURIAS,127,"4,3","2,6","2,0","6,6","0,0",...,01:00,99.0,"7,2",00:13,NaN,NaN,NaN,NaN,-6.044167,43.566944
4,1970-01-01,0016A,REUS AEROPUERTO,TARRAGONA,71,"5,5","0,0","0,6","10,4","1,7",...,14:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.163611,41.145
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7427566,2024-12-31,9562X,MORELLA,CASTELLON,990,"4,8","0,0","0,8","8,8","1,4",...,13:42,12.0,"5,0",16:50,97.0,23:50,63.0,14:00,-0.101944,40.621667
7427567,2024-12-31,1002Y,"BAZTAN, IRURITA",NAVARRA,183,"4,2","0,0","-4,9","13,4","0,0",...,14:50,22.0,"3,1",16:20,100.0,09:00,50.0,14:10,-1.543056,43.135833
7427568,2024-12-31,3254Y,MORA,TOLEDO,717,"2,0","0,0","-3,1","7,1","1,1",...,11:18,20.0,"3,9",14:30,100.0,09:20,80.0,11:20,-3.780556,39.686944
7427569,2024-12-31,0194D,"CORBERA, PUIG D'AGULLES",BARCELONA,647,NaN,"0,0",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.885,41.408056


### Preprocesado del Dataframe

In [6]:
df_prep = df.copy(deep = True)

In [7]:
COL_PREC = 'prec'
# df_prep.loc[df[COL_PREC] == 'Ip', COL_PREC] = 0
df_prep = df_prep[df_prep[COL_PREC] != 'Acum']
df_prep[COL_PREC] = df_prep[COL_PREC].replace('Ip', 0)

In [8]:
# Reemplazamos las comas por puntos para poder cargar los números como decimales (floats)
df_prep[COLS_FLOATS] = df_prep[COLS_FLOATS].replace(',', '.', regex=True).apply(pd.to_numeric, errors = 'raise')

In [9]:
df_prep.dtypes

fecha          datetime64[ns]
indicativo     string[python]
nombre         string[python]
provincia      string[python]
altitud                 Int64
tmed                  float64
prec                  float64
tmin                  float64
tmax                  float64
velmedia              float64
sol                   float64
presMax               float64
horaPresMax            object
presMin               float64
horaPresMin            object
hrMedia               float64
horatmin               object
horatmax               object
dir                   float64
racha                 float64
horaracha              object
hrMax                 float64
horaHrMax              object
hrMin                 float64
horaHrMin              object
lon                   Float64
lat                   Float64
dtype: object

In [10]:
# Corregimos inconsistencias en nombres de provincias
df_prep.loc[df_prep['provincia'] == 'BALEARES', 'provincia'] = 'ILLES BALEARS'
df_prep.loc[df_prep['provincia'] == 'SANTA CRUZ DE TENERIFE', 'provincia'] = 'STA. CRUZ DE TENERIFE'

In [11]:
# Borramos dos datos de una medición anómala
df_prep.loc[
    (df_prep['indicativo'] == '6084X') &
    (df_prep['tmin'] == 50.0) &
    (df_prep['tmax'] == -50.0),
    ['tmax', 'tmin']
] = np.nan

In [12]:
# Procesamos dirección del viento (según la info proporcionada en el fichero de metadatos)
dict_dir = {
    88: 'Desconocida',
    99: 'Varias'
}
dir_por_defecto = 'Válida'

df_prep['dir_tipo'] = df_prep['dir'].map(dict_dir).fillna(dir_por_defecto)

In [13]:
# Añadimos el sufijo :00 a las columnas cuyos valores carecen de él
cols_sufijo = ['horaPresMax', 'horaPresMin']
df_prep[cols_sufijo] = df_prep[cols_sufijo].where(
    df_prep[cols_sufijo].isna() | (df_prep[cols_sufijo] == 'Varias'),
    df_prep[cols_sufijo] + ':00'
)

In [14]:
df_prep.sample(20)

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,tmax,velmedia,...,dir,racha,horaracha,hrMax,horaHrMax,hrMin,horaHrMin,lon,lat,dir_tipo
556655,1982-09-01,1387,A CORUÑA,A CORUÑA,57,17.1,0.0,15.0,19.2,5.3,...,2.0,11.7,17:30,NaN,NaN,NaN,NaN,-8.421389,43.365833,Válida
789688,1987-05-17,8019,ALICANTE-ELCHE AEROPUERTO,ALICANTE,43,17.6,0.0,12.0,23.2,3.3,...,18.0,12.8,13:25,NaN,NaN,NaN,NaN,-0.570833,38.282778,Válida
6981126,2023-08-15,9689X,TORRE DE CABDELLA,LLEIDA,1273,21.5,0.0,14.3,28.7,NaN,...,NaN,NaN,NaN,89.0,02:40,33.0,13:40,0.990833,42.466389,Válida
2698549,2009-02-24,1442U,BOIRO,A CORUÑA,10,11.2,0.0,3.5,19.0,NaN,...,NaN,NaN,NaN,99.0,09:05,46.0,Varias,-8.892222,42.643889,Válida
4705576,2016-05-09,2430Y,MUÑOTELLO,AVILA,1178,8.6,9.2,6.1,11.1,3.6,...,14.0,18.3,14:00,88.0,02:50,70.0,16:30,-5.044167,40.543333,Válida
5260080,2018-02-24,9619,LA SEU D'URGELL,LLEIDA,677,3.3,0.0,-5.8,12.4,NaN,...,NaN,NaN,NaN,95.0,Varias,29.0,14:00,1.462778,42.354722,Válida
1765149,2001-08-01,5530E,GRANADA AEROPUERTO,GRANADA,560,29.3,0.0,21.4,37.2,2.2,...,29.0,7.8,15:20,NaN,NaN,NaN,NaN,-3.789722,37.190278,Válida
7349726,2024-10-04,1026X,ORDIZIA,GIPUZKOA,290,14.0,0.0,9.7,18.4,1.7,...,13.0,5.8,13:40,100.0,Varias,63.0,13:40,-2.184167,43.058056,Válida
1210737,1994-05-04,C429I,TENERIFE SUR AEROPUERTO,STA. CRUZ DE TENERIFE,64,20.3,0.0,17.0,23.6,8.3,...,7.0,12.8,Varias,NaN,NaN,NaN,NaN,-16.561111,28.046944,Válida
3252066,2011-05-10,0002I,VANDELLÒS,TARRAGONA,32,NaN,NaN,NaN,NaN,1.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.871389,40.958056,Válida


In [15]:
df_prep.to_csv('Valores_Climatologicos_1970_2024_Limpios.csv', sep = ';', index = False)